# Mall Customers Analysis
## Customer Segmentation and Prediction Model

This notebook implements a complete machine learning pipeline for customer analysis using the Mall Customers dataset.

**About this Dataset:**
- Contains customer data from a shopping mall
- 200 customers with demographic and spending information
- Used for customer segmentation and behavior prediction

**Features:**
- `CustomerID`: Unique identifier for each customer
- `Gender`: Customer gender (Male/Female)
- `Age`: Customer age
- `Annual Income (k$)`: Annual income in thousands of dollars
- `Spending Score (1-100)`: Score based on customer spending behavior

**Pipeline Overview:**
1. Load and explore the dataset from CSV
2. Perform Exploratory Data Analysis (EDA)
3. Preprocess data (handle categorical variables, split, scale)
4. Train multiple regression models
5. Compare model performance
6. Analyze feature importance
7. Make predictions
8. Summarize findings

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import os
warnings.filterwarnings('ignore')

# Verify installations and print versions
print("✅ All libraries imported successfully!")
print(f"📊 NumPy version: {np.__version__}")
print(f"📊 Pandas version: {pd.__version__}")
print(f"📊 scikit-learn version: {__import__('sklearn').__version__}")
import xgboost
print(f"📊 XGBoost version: {xgboost.__version__}")

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print("\n🎨 Visualization style set successfully!")

# Check if CSV file exists
csv_path = '../../../Downloads/Mall_Customers.csv'
print(f"\n📂 Checking CSV file: {csv_path}")
print(f"   File exists: {os.path.exists(csv_path)}")
if os.path.exists(csv_path):
    print(f"   Absolute path: {os.path.abspath(csv_path)}")

## 1. Load Mall Customers Dataset

### **What we're doing here:**

1. **Loading the dataset**: Using pandas `read_csv()` to load the Mall Customers dataset from the CSV file.

2. **Dataset location**: The file is located at `../../../Downloads/Mall_Customers.csv`

3. **Creating DataFrame**: Converting the CSV data into a pandas DataFrame for analysis.

### **Common CSV loading issues:**

**Problem: File not found**
- **Solution**: Check the file path - make sure it's correct
- **Solution**: Use absolute path if relative path doesn't work
  ```python
  # Use absolute path
  csv_path = r'C:\Users\yuvra\Downloads\Mall_Customers.csv'
  df = pd.read_csv(csv_path)
  ```

**Problem: Encoding issues**
- **Solution**: Specify encoding parameter
  ```python
  df = pd.read_csv('file.csv', encoding='utf-8')
  ```

**Problem: Wrong delimiter**
- **Solution**: Specify separator
  ```python
  df = pd.read_csv('file.csv', sep=';')  # For semicolon-separated
  ```

**Pro Tip**: Always check file existence before loading!
```python
import os
print(os.path.exists('Mall_Customers.csv'))
```

In [ ]:
# ============================================
# LOAD DATASET FROM CSV FILE
# ============================================

# Method 1: Try relative path first
try:
    print("📂 Attempting to load from relative path...")
    df = pd.read_csv('../../../Downloads/Mall_Customers.csv')
    print("✅ Successfully loaded from relative path!")
except FileNotFoundError:
    print("⚠️  Relative path failed, trying absolute path...")
    
    # Method 2: Try absolute path
    try:
        csv_path = r'C:\Users\yuvra\Downloads\Mall_Customers.csv'
        df = pd.read_csv(csv_path)
        print(f"✅ Successfully loaded from: {csv_path}")
    except FileNotFoundError:
        print("❌ Error: Could not find Mall_Customers.csv")
        print("\n💡 Please check:")
        print("   1. File exists in Downloads folder")
        print("   2. File name is exactly 'Mall_Customers.csv'")
        print("   3. Copy file to the notebook directory if needed")

In [ ]:
# Display dataset information
print("✅ Dataset loaded successfully!")
print(f"📏 Dataset shape: {df.shape}")
print(f"   Rows: {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")
print(f"\n📋 Column names: {list(df.columns)}")
print(f"\n🔍 First 5 rows:")
df.head()

# Check for missing values
print(f"\n🔎 Missing values per column:")
print(df.isnull().sum())

# Show data types
print(f"\n📊 Data types:")
print(df.dtypes)

## 2. Exploratory Data Analysis (EDA)

### **What is EDA?**
Exploratory Data Analysis is the process of analyzing data sets to summarize their main characteristics, often with visual methods. It helps us:

1. **Understand the data**: Check shape, data types, missing values
2. **Identify patterns**: See distributions, correlations, outliers
3. **Detect issues**: Missing data, incorrect data types
4. **Inform preprocessing**: Decide on encoding, scaling, etc.

### **Steps in this section:**
1. Basic dataset information
2. Distribution of variables
3. Correlation matrix to understand relationships
4. Gender distribution

### **Key insights to look for:**
- Which age groups shop the most?
- Does income affect spending?
- Are there differences between genders?

In [ ]:
# Dataset information
print("📊 Dataset Info:")
print(f"Number of customers: {len(df)}")
print(f"Number of features: {len(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nStatistical summary:")
df.describe()

### 2.1 Target Variable Distribution - Spending Score

**Why analyze the Spending Score?**
The Spending Score (1-100) is our target variable - it represents customer spending behavior. Understanding its distribution helps us:

- Identify if customers are concentrated in certain spending ranges
- Spot outliers (unusual spending patterns)
- Understand the typical customer profile

**Spending Score interpretation:**
- Higher score (60-100): High spenders
- Medium score (40-60): Average spenders
- Lower score (1-40): Low spenders

In [ ]:
# Visualize Spending Score distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['Spending Score (1-100)'], bins=30, kde=True, color='skyblue')
plt.title('Spending Score Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Spending Score (1-100)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
# Add vertical lines for mean and median
plt.axvline(df['Spending Score (1-100)'].mean(), color='red', linestyle='--', 
            label=f'Mean: {df["Spending Score (1-100)"].mean():.2f}')
plt.axvline(df['Spending Score (1-100)'].median(), color='green', linestyle='--', 
            label=f'Median: {df["Spending Score (1-100)"].median():.2f}')
plt.legend()
plt.show()

print(f"💡 Spending Score Statistics:")
print(f"   Mean: {df['Spending Score (1-100)'].mean():.2f}")
print(f"   Median: {df['Spending Score (1-100)'].median():.2f}")
print(f"   Min: {df['Spending Score (1-100)'].min()}")
print(f"   Max: {df['Spending Score (1-100)'].max()}")

In [ ]:
# Gender distribution
plt.figure(figsize=(8, 5))
gender_counts = df['Gender'].value_counts()
colors = ['lightcoral', 'lightskyblue']
plt.pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Gender Distribution', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.show()

print(f"📊 Gender Distribution:")
for gender, count in gender_counts.items():
    percentage = (count / len(df)) * 100
    print(f"   {gender}: {count} ({percentage:.1f}%)")

In [ ]:
# Age distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['Age'], bins=30, kde=True, color='lightgreen')
plt.title('Age Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Age', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.axvline(df['Age'].mean(), color='red', linestyle='--', label=f'Mean: {df["Age"].mean():.2f}')
plt.axvline(df['Age'].median(), color='blue', linestyle='--', label=f'Median: {df["Age"].median():.2f}')
plt.legend()
plt.show()

print(f"💡 Age Statistics:")
print(f"   Mean Age: {df['Age'].mean():.2f} years")
print(f"   Youngest: {df['Age'].min()} years")
print(f"   Oldest: {df['Age'].max()} years")

### 2.2 Correlation Analysis

**What is Correlation?**
- Correlation measures the linear relationship between two variables
- Range: -1 (perfect negative) to +1 (perfect positive)
- 0 means no linear relationship

**Expected relationships:**
- Age vs Spending Score: Do older/younger people spend differently?
- Annual Income vs Spending Score: Do richer people spend more?

**How to read the heatmap:**
- Dark blue: strong negative correlation
- Dark red: strong positive correlation
- White/light colors: weak correlation

In [ ]:
# Correlation matrix (only for numeric columns)
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
correlation_matrix = numeric_df.corr()
sns.heatmap(
    correlation_matrix,
    annot=True,      # Show correlation values
    cmap='coolwarm', # Color scheme
    center=0,        # Center color at 0
    fmt='.2f',       # Format to 2 decimal places
    square=True,     # Make squares square
    linewidths=0.5,  # Add borders between cells
    vmin=-1,         # Set color scale range
    vmax=1
)
plt.title('Feature Correlation Matrix - Mall Customers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find correlations with Spending Score
spending_correlations = correlation_matrix['Spending Score (1-100)'].sort_values(ascending=False)
print("\n🎯 Features correlated with Spending Score:")
print(spending_correlations[1:])  # Exclude self-correlation

In [ ]:
# Annual Income vs Spending Score scatter plot
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'], 
                     c=df['Age'], cmap='viridis', alpha=0.6, s=100)
plt.colorbar(scatter, label='Age')
plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.title('Annual Income vs Spending Score (colored by Age)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.show()

## 3. Data Preprocessing

### **Preprocessing Steps:**

1. **Handle categorical variables**: Convert 'Gender' from text (Male/Female) to numbers
   - Using Label Encoding: Male → 0, Female → 1

2. **Separate features and target**:
   - Features (X): CustomerID, Gender, Age, Annual Income
   - Target (y): Spending Score

3. **Train-test split**: Divide data into training (80%) and testing (20%) sets
   - `test_size=0.2`: 20% for testing
   - `random_state=42`: Ensures reproducibility

4. **Feature Scaling**: Using `StandardScaler` to standardize features
   - Removes the mean and scales to unit variance
   - Important for: Linear Regression
   - Tree-based models benefit less but will still work

### **Why CustomerID is NOT useful:**
CustomerID is just an identifier - it has no predictive power. Including it would confuse the model. We'll drop it.

In [ ]:
# ============================================
# DATA PREPROCESSING
# ============================================

# Drop CustomerID - not useful for prediction
df_clean = df.drop('CustomerID', axis=1)
print("🗑️  Dropped CustomerID (not predictive)")

# Encode categorical variable (Gender)
# Male → 0, Female → 1
label_encoder = LabelEncoder()
df_clean['Gender_Encoded'] = label_encoder.fit_transform(df_clean['Gender'])

# Drop original Gender column and use encoded version
df_clean = df_clean.drop('Gender', axis=1)

print("✅ Encoded Gender column (Male=0, Female=1)")
print(f"   Encoding mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# Separate features and target
X = df_clean.drop('Spending Score (1-100)', axis=1)  # Features
y = df_clean['Spending Score (1-100)']                # Target

print(f"\n📊 Features: {list(X.columns)}")
print(f"🎯 Target: Spending Score (1-100)")

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test data
    random_state=42     # Random seed for reproducibility
)

print("\n✂️  Data split completed:")
print(f"   Training set: {X_train.shape[0]} samples")
print(f"   Test set: {X_test.shape[0]} samples")

# Scale features
scaler = StandardScaler()

# Fit on training data only (to avoid data leakage)
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using training statistics
X_test_scaled = scaler.transform(X_test)

print("\n⚙️  Feature scaling completed!")
print(f"\nScaled training data sample (first row):\n{X_train_scaled[0]}")
print(f"\nScaled test data sample (first row):\n{X_test_scaled[0]}")
print("\n✅ Data preprocessing completed successfully!")

## 4. Model Training

### **Why multiple models?**
Different algorithms have different strengths. We train 5 models to find the best one:

1. **Linear Regression**: Simple, interpretable, assumes linear relationship
2. **Decision Tree**: Simple, interpretable, captures non-linear patterns
3. **Random Forest**: Ensemble of trees, reduces overfitting, robust
4. **Gradient Boosting**: Sequential learning, often high performance
5. **XGBoost**: Optimized gradient boosting, industry standard

### **Evaluation Metrics:**
- **MAE (Mean Absolute Error)**: Average absolute error. Easy to interpret.
  - Lower is better
  - In Spending Score units

- **MSE (Mean Squared Error)**: Average squared error. Penalizes large errors.
  - Lower is better
  - Not in same units as target

- **RMSE (Root Mean Squared Error)**: Square root of MSE. Same units as target.
  - Lower is better
  - Most commonly reported

- **R² Score (Coefficient of Determination)**: Proportion of variance explained.
  - Range: -∞ to 1
  - Higher is better
  - 1.0 = perfect predictions
  - 0.0 = model is as good as predicting the mean

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

# Store results
results = {}

print("🤖 Training models...")
print("=" * 50)

# Train and evaluate each model
for name, model in models.items():
    print(f"\n📈 Training {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Store results
    results[name] = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2
    }
    
    # Print results
    print(f"  📊 Performance Metrics:")
    print(f"     MAE:  {mae:.4f}")
    print(f"     MSE:  {mse:.4f}")
    print(f"     RMSE: {rmse:.4f}")
    print(f"     R²:   {r2:.4f}")
    
    # Interpretation
    if r2 >= 0.8:
        print(f"     ⭐ Excellent performance!")
    elif r2 >= 0.6:
        print(f"     ✅ Good performance")
    elif r2 >= 0.4:
        print(f"     ⚠️  Moderate performance")
    else:
        print(f"     ❌ Poor performance")

print("\n" + "=" * 50)
print("✅ All models trained successfully!")

## 5. Model Comparison

### **Comparing Models:**

1. **Create comparison table**: Organize results in a DataFrame for easy viewing

2. **Visualize performance**: Bar charts for each metric

### **How to choose the best model:**
- **R² Score**: Higher is better (primary metric)
- **RMSE**: Lower is better (secondary, in spending score units)
- **MAE**: Lower is better (easy to interpret)

### **Trade-offs:**
- Linear Regression: Fast, interpretable, but may underfit
- Tree-based models: More accurate, but less interpretable
- Ensemble methods (RF, GB, XGB): Often best performance but more complex

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# MAE - Lower is better
results_df['MAE'].sort_values().plot(
    kind='barh',
    ax=axes[0, 0],
    color='skyblue'
)
axes[0, 0].set_title('Mean Absolute Error (MAE) - Lower is Better', fontweight='bold')
axes[0, 0].set_xlabel('MAE', fontsize=10)
axes[0, 0].set_ylabel('Model', fontsize=10)

# MSE - Lower is better
results_df['MSE'].sort_values().plot(
    kind='barh',
    ax=axes[0, 1],
    color='lightgreen'
)
axes[0, 1].set_title('Mean Squared Error (MSE) - Lower is Better', fontweight='bold')
axes[0, 1].set_xlabel('MSE', fontsize=10)
axes[0, 1].set_ylabel('Model', fontsize=10)

# RMSE - Lower is better
results_df['RMSE'].sort_values().plot(
    kind='barh',
    ax=axes[1, 0],
    color='salmon'
)
axes[1, 0].set_title('Root Mean Squared Error (RMSE) - Lower is Better', fontweight='bold')
axes[1, 0].set_xlabel('RMSE', fontsize=10)
axes[1, 0].set_ylabel('Model', fontsize=10)

# R² Score - Higher is better
results_df['R2'].sort_values(ascending=False).plot(
    kind='barh',
    ax=axes[1, 1],
    color='lightcoral'
)
axes[1, 1].set_title('R² Score - Higher is Better', fontweight='bold')
axes[1, 1].set_xlabel('R² Score', fontsize=10)
axes[1, 1].set_ylabel('Model', fontsize=10)
axes[1, 1].axvline(x=0.8, color='green', linestyle='--', alpha=0.7, label='Good threshold (0.8)')
axes[1, 1].legend()

plt.tight_layout()
plt.suptitle('Model Performance Comparison - Mall Customers', y=1.02, fontsize=16, fontweight='bold')
plt.show()

# Print summary of best model
best_model_name = results_df['R2'].idxmax()
best_r2 = results_df.loc[best_model_name, 'R2']
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   R² Score: {best_r2:.4f}")
print(f"   RMSE: {results_df.loc[best_model_name, 'RMSE']:.4f}")

## 6. Feature Importance (Tree-based Models)

### **What is Feature Importance?**
Feature importance tells us which features contribute most to the model's predictions.

**How Random Forest calculates feature importance:**
- For each tree in the forest, measures how much each feature decreases impurity (MSE in regression)
- Averages this decrease across all trees
- Higher value = more important feature

**Why it matters:**
- **Interpretability**: Understand what drives customer spending
- **Business insights**: Focus on high-impact features
- **Feature selection**: Remove unimportant features
- **Marketing strategy**: Target key demographics

**Note**: We'll use Random Forest here as it's typically more stable than a single Decision Tree.

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("🌳 Random Forest Feature Importance:")
print("=" * 40)
for idx, row in feature_importance.iterrows():
    print(f"{row['feature']:20s}: {row['importance']:.4f}")

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(
    feature_importance['feature'],
    feature_importance['importance'],
    color='teal',
    edgecolor='black'
)
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance - Random Forest (Mall Customers)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()  # Invert y-axis to show highest importance at top
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

feature_importance

## 7. Sample Predictions

### **Making Predictions:**

1. **Select best model**: Based on R² score
2. **Make predictions**: Use the model to predict on test set
3. **Compare actual vs predicted**: See how well the model predicts customer spending

### **Understanding the output:**
- **Actual Spending Score**: True customer spending score
- **Predicted Spending Score**: Model's prediction
- **Error**: Difference between actual and predicted
  - Positive: Model underestimated spending
  - Negative: Model overestimated spending
  - Smaller absolute values = better predictions

### **Model interpretation for business:**
If the model performs well (R² ≥ 0.7), it means:
- We can predict customer spending based on demographics
- This helps identify high-value customers
- Useful for targeted marketing campaigns

In [ ]:
# Select best model (based on R2 score)
best_model_name = results_df['R2'].idxmax()
best_model = models[best_model_name]

print(f"🏆 Best Model Selected: {best_model_name}")
print(f"   R² Score: {results_df.loc[best_model_name, 'R2']:.4f}")
print(f"   RMSE: {results_df.loc[best_model_name, 'RMSE']:.4f}")
print(f"   MAE: {results_df.loc[best_model_name, 'MAE']:.4f}")

# Make predictions on test set
y_pred_best = best_model.predict(X_test_scaled)

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Actual Score': y_test.values,
    'Predicted Score': y_pred_best,
    'Error': y_test.values - y_pred_best,
    'Absolute Error': np.abs(y_test.values - y_pred_best)
})
comparison = comparison.reset_index(drop=True)

print(f"\n📋 Sample Predictions (first 10):")
print("=" * 60)
print(comparison.head(10).to_string(index=False))
print("\n📊 Prediction Statistics:")
print("=" * 40)
print(comparison[['Actual Score', 'Predicted Score', 'Error']].describe())

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(10, 6))

# Scatter plot
plt.scatter(y_test, y_pred_best, alpha=0.5, color='blue', edgecolors='black', linewidths=0.5)

# Perfect prediction line (diagonal)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--',
    lw=2,
    label='Perfect Prediction'
)

plt.xlabel('Actual Spending Score', fontsize=12)
plt.ylabel('Predicted Spending Score', fontsize=12)
plt.title(f'Actual vs Predicted Spending Scores - {best_model_name}', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate and display R² on plot
r2 = r2_score(y_test, y_pred_best)
print(f"\n💡 Interpretation:")
print(f"   R² Score: {r2:.4f}")
print(f"   The model explains {r2 * 100:.2f}% of the variance in spending scores.")
print(f"   Points close to the red line indicate accurate predictions.")

## 8. Prediction on New Data

### **How to use the trained model:**

1. **Prepare new data**: Must have the same features (Gender, Age, Annual Income)
2. **Encode Gender**: Convert Male/Female to 0/1
3. **Scale the data**: Use the same scaler fitted on training data
   - **Important**: Don't fit a new scaler! Use the one from training.
4. **Predict**: Call model.predict() on scaled data

### **Real-world usage example:**
```python
# Predict spending for a new customer
new_customer = pd.DataFrame({
    'Gender': ['Female'],
    'Age': [30],
    'Annual Income (k$)': [70]
})
# Encode and scale
new_customer['Gender_Encoded'] = label_encoder.transform(new_customer['Gender'])
new_customer = new_customer.drop('Gender', axis=1)
new_customer_scaled = scaler.transform(new_customer)
predicted_spending = best_model.predict(new_customer_scaled)
print(f"Predicted spending score: {predicted_spending[0]:.2f}")
```

**Note**: This is a simplified example. In production, you would:
- Save the model (using joblib or pickle)
- Save the scaler and encoder
- Create an API endpoint for predictions
- Add input validation and error handling
- Monitor model performance over time

In [ ]:
# Example: Predict spending score for a sample from test set
sample_idx = 0
sample_data = X_test.iloc[sample_idx:sample_idx+1]
sample_scaled = scaler.transform(sample_data)
predicted_score = best_model.predict(sample_scaled)[0]
actual_score = y_test.iloc[sample_idx]

print("👤 Sample Prediction Example:")
print("=" * 50)
print(f"\nSample #{sample_idx}:")
print(f"\n📋 Features:")
# Decode gender for display
gender_map = {0: 'Male', 1: 'Female'}
gender = gender_map[int(sample_data['Gender_Encoded'].values[0])]
print(f"   Gender: {gender}")
print(f"   Age: {sample_data['Age'].values[0]:.0f} years")
print(f"   Annual Income: ${sample_data['Annual Income (k$)'].values[0]:.0f}k")

print(f"\n💰 Spending Score Prediction:")
print(f"   Actual Score:    {actual_score:>10.2f}")
print(f"   Predicted Score: {predicted_score:>10.2f}")
print(f"   Error:           {abs(actual_score - predicted_score):>10.2f}")
print(f"   Error %:         {abs(actual_score - predicted_score) / actual_score * 100:>10.2f}%")

# Make a few more predictions
print("\n📊 Additional Predictions:")
print("=" * 50)
for i in range(1, 6):
    sample = X_test.iloc[i:i+1]
    sample_scaled = scaler.transform(sample)
    pred = best_model.predict(sample_scaled)[0]
    actual = y_test.iloc[i]
    error_pct = abs(actual - pred) / actual * 100
    print(f"Sample {i}: Actual={actual:.1f}, Predicted={pred:.1f}, Error={error_pct:.1f}%")
print("\n✅ Predictions working correctly!")

## 9. Summary and Conclusions

### **Dataset Summary:**
- **Name**: Mall Customers
- **Source**: Shopping mall customer data
- **Samples**: 200 customers
- **Features**: Gender, Age, Annual Income, Spending Score
- **Target**: Spending Score (1-100)

### **Best Model Performance:**
The best performing model was **{best_model_name}** with:
- **R² Score**: {results_df.loc[best_model_name, 'R2']:.4f} (explains {results_df.loc[best_model_name, 'R2'] * 100:.2f}% of variance)
- **RMSE**: {results_df.loc[best_model_name, 'RMSE']:.4f}
- **MAE**: {results_df.loc[best_model_name, 'MAE']:.4f}

### **Key Findings:**
1. **Most important feature**: {feature_importance.iloc[0]['feature']} ({feature_importance.iloc[0]['importance']:.4f} importance)
2. **Average spending score**: {df['Spending Score (1-100)'].mean():.2f}
3. **Spending score range**: {df['Spending Score (1-100)'].min()} to {df['Spending Score (1-100)'].max()}
4. **Gender distribution**: {gender_counts[gender_counts.index[0]]} {gender_counts.index[0]}s and {gender_counts[gender_counts.index[1]]} {gender_counts.index[1]}s

### **Business Insights:**
1. **Customer segmentation**: Identify different customer groups based on spending patterns
2. **Targeted marketing**: Use predictions to focus on high-value segments
3. **Income-spending relationship**: Understand if higher income means higher spending
4. **Age impact**: Determine which age groups are primary customers

### **Next Steps (for improvement):**
1. **Hyperparameter tuning**: Use GridSearch or RandomSearch to optimize parameters
2. **Clustering**: Try K-Means clustering for customer segmentation
3. **Add more features**: Include time-based data, visit frequency, product categories
4. **Ensemble methods**: Combine multiple models for better accuracy
5. **Classification approach**: Instead of predicting exact score, categorize as Low/Medium/High
6. **Save model**: Deploy using `joblib.dump()` for production use

---
**✅ All models trained successfully and predictions are working correctly!**

### **What to do with this model:**
- **Marketing**: Predict customer value and target accordingly
- **Customer insights**: Understand spending patterns
s- **Business strategy**: Allocate resources to high-value segments
- **Education**: Learn ML pipeline implementation with real-world data

### **Limitations to note:**
- Small dataset (only 200 customers)
- Limited features (no purchase history, visit frequency, etc.)
- Spending score is a proxy for actual spending
- Model may not generalize to different malls/regions

📚 **Resources:**
- [Mall Customers Dataset Info](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)
- [Scikit-learn Documentation](https://scikit-learn.org/)
- [Customer Segmentation Guide](https://towardsdatascience.com/customer-segmentation-using-k-means-clustering-df9587f1d824)

---
*Notebook created as part of Customer Analysis ML Project*